In [0]:
sales_customers_df = spark.read.table("samples.bakehouse.sales_customers")

display(sales_customers_df)

In [0]:
%sql
DESCRIBE HISTORY samples.bakehouse.sales_customers;
DESCRIBE EXTENDED samples.bakehouse.sales_customers;

In [0]:
sales_customers_df.write.mode("append").saveAsTable("practice_catalog.default.sales_customers_dups")

In [0]:
%sql
DESCRIBE HISTORY practice_catalog.default.sales_customers_dups;

In [0]:
%sql
SELECT * from practice_catalog.default.sales_customers_dups;
SELECT DISTINCT * from practice_catalog.default.sales_customers_dups;

In [0]:
sales_customers_df.printSchema()
print(sales_customers_df.schema)

### How to Load CSV Files

In [0]:
csv_df = spark.read.csv("/Volumes/dbacademy/default/sample_files/baby_names.csv")
display(csv_df.limit(3))

In [0]:
csv_1_df = spark.read.csv("/Volumes/dbacademy/default/sample_files/baby_names.csv", header=True, inferSchema=True)
display(csv_1_df.limit(3))

In [0]:
csv_2_df = spark.read.format("csv").load("/Volumes/dbacademy/default/sample_files/baby_names.csv")
display(csv_2_df.limit(3))

In [0]:
csv_3_df = spark.read.format("csv").load("/Volumes/dbacademy/default/sample_files/baby_names.csv" , header=True, inferSchema=True)
display(csv_3_df.limit(3))

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("State", StringType(), True),
    StructField("Sex", StringType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Count", IntegerType(), True)
])

csv_4_df = spark.read.format("csv").schema(schema).load("/Volumes/dbacademy/default/sample_files/baby_names_2.csv")
display(csv_4_df.limit(3))

In [0]:
schema = "Id INT, State STRING, Sex STRING, Year INT, Name STRING, Count INT"

csv_5_df = spark.read.format("csv").schema(schema).load("/Volumes/dbacademy/default/sample_files/baby_names_2.csv")
display(csv_5_df.limit(3))

In [0]:
csv_6_df = spark.read.format("csv").load("/Volumes/dbacademy/default/tutorials/winequality-red.csv" , header=True, sep=";" , inferSchema=True)
display(csv_6_df.limit(3))

### How to Load JSON Files

#### Single-Line Vs Multi-Line JSON

The difference isn't about the JSON key-value syntax itself—it is about **how files are formatted across lines** and **how distributed engines like Apache Spark read and split the data**.

---

##### Core Structural Difference

**1. Single-Line JSON (JSON Lines / NDJSON / JSONL)**

* **Format:** Every complete JSON object occupies strictly **one physical line**, separated by a newline character `\n`.
* **Example:**
```json
{"id": 1, "name": "Dipesh", "role": "Data Engineer"}
{"id": 2, "name": "Amit", "role": "Cloud Architect"}
{"id": 3, "name": "Rohan", "role": "Data Analyst"}

```

**2. Multi-Line JSON (Standard Pretty-Printed / Nested JSON Array)**

* **Format:** A single JSON object or array is formatted across **multiple lines** with indentation and whitespace for human readability.
* **Example:**
```json
[
  {
    "id": 1,
    "name": "Dipesh",
    "role": "Data Engineer"
  },
  {
    "id": 2,
    "name": "Amit",
    "role": "Cloud Architect"
  }
]

```

---

##### Comparison: Single-Line vs. Multi-Line JSON

| Feature | Single-Line JSON (`JSONL` / `NDJSON`) | Multi-Line JSON (Standard Pretty JSON) |
| --- | --- | --- |
| **Record Delimiter** | Newline `\n` defines each individual record. | Brackets `[` `]`, braces `{` `}`, and commas `,`. |
| **Spark Default Behavior** | Spark's **default** mode (`multiLine=false`). | Requires explicit flag: `.option("multiLine", "true")`. |
| **File Splittability** | **Splittable:** Spark executors can split a large file at line breaks and read chunks in parallel. | **Non-splittable:** A single executor/core must read and parse the entire file from start to end. |
| **Distributed Performance** | **High:** Fully parallelized ingestion across Spark cluster nodes. | **Low:** Bottlenecks ingestion; single-threaded parse step per file. |
| **Memory Consumption** | **Low:** Records stream row-by-row into memory. | **High:** Can trigger Out-of-Memory (OOM) errors on large files because the full JSON tree must be loaded to parse root brackets. |
| **Use Cases** | Log streams, IoT telemetry, high-throughput Big Data ETL. | API payloads, configuration files, human-edited documents. |

---

##### PySpark Code Comparison

```python
# 1. Reading Single-Line JSON (Default & Highly Optimized)
df_single = spark.read.json("path/to/single_line_data.json")

# 2. Reading Multi-Line JSON (Requires multiLine flag)
df_multi = spark.read.option("multiLine", "true").json("path/to/multi_line_data.json")

```
---

#### Interview Takeaway

If asked in an interview:

> *"Why is single-line JSON preferred over multi-line JSON in Big Data pipelines?"*

**Answer:**

In Single-Line JSON, each line is an independent, valid JSON record. This allows Spark to split massive multi-gigabyte files across multiple worker nodes to parse in parallel. In Multi-Line JSON, Spark cannot determine record boundaries by line breaks alone, so it must parse the file as a single entity on one core with `multiLine=true`, creating an I/O bottleneck and increasing OOM risk.

In [0]:
single_line_json_df = spark.read.json("/Volumes/dbacademy/default/sample_files/employees.json")
display(single_line_json_df.limit(3))

In [0]:
multi_line_json_df = spark.read.json("/Volumes/dbacademy/default/sample_files/employee.json" , multiLine=True)
display(multi_line_json_df.limit(3))

In [0]:
# Save single-line JSON DataFrame as CSV
single_line_json_df.write.format("csv").mode("overwrite").save("/Volumes/dbacademy/default/sample_files/outputs/csv_files/")

# Save multi-line JSON DataFrame as JSON
multi_line_json_df.write.format("json").mode("overwrite").save("/Volumes/dbacademy/default/sample_files/outputs/json_files/")

### How to Load Parquet Files

In [0]:
display(dbutils.fs.ls("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF3/"))
display(dbutils.fs.ls("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF3/ServiceName=Azure Databricks/"))

In [0]:
parquet_df = spark.read.parquet("/Volumes/dbacademy/default/tutorials/azure_usage_cost_per_service_PF3/")
display(parquet_df)

### Extra

In [0]:
# Load Spark DataFrame from Delta Lake table
sample_df = spark.read.table("dbacademy.default.winequality_red")

# Display DataFrame in Databricks notebook
display(sample_df)

# Save DataFrame as CSV files with 5 partitions
sample_df.repartition(5).write.format("csv").mode("overwrite").save("/Volumes/dbacademy/default/tutorials/winequality_red_CSVs")

In [0]:
# Define schema for winequality_red CSV files
schema = "wine_id LONG, fixed_acidity DOUBLE, volatile_acidity DOUBLE, citric_acid DOUBLE, residual_sugar DOUBLE, chlorides DOUBLE, free_sulfur_dioxide DOUBLE, total_sulfur_dioxide DOUBLE, density DOUBLE, pH DOUBLE, sulphates DOUBLE, alcohol DOUBLE, quality INT"

# Load CSV files into Spark DataFrame using defined schema
winequality_red_csv_df = spark.read.format("csv").load("/Volumes/dbacademy/default/tutorials/winequality_red_CSVs/" , schema=schema)

# Display DataFrame
display(winequality_red_csv_df)